In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP03 - TP Services
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Did the unit use third party intermediaries or subcontractors during the assessment 
   period to perform marketing, business development, sales, consulting, or brokerage?
   
   LOGIC SUMMARY (Original & Updated Architecture):
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. RISK FLAG LOGIC: Flags engagements as high risk if they meet either condition:
      - KPI_Number = '8.5' AND KPI_Value = '100' (Updated from 'Value' column)
      - primary_product_serv matches one of the 14 predefined marketing/printing categories.
        (Updated to be Case-Insensitive using UPPER).
   3. COMMA EXCEPTION HANDLING: Protects specific unit names containing internal commas 
      (e.g., 'CAD PB - RESL, Everyday Banking...') by replacing them with '[COMMA]' 
      so the split function doesn't shred them.
   4. FLATTEN LOBs: Columns K (owning_lob) and L (lob_using) contain comma-separated 
      lists. Uses LATERAL VIEW explode(split(...)) to expand these into individual rows.
   5. TPRM STRING MAPPING: Maps flagged TP engagements to AU IDs by checking if the 
      'TPRM_Units' string from the mapping table exists inside the text.
      - Blocks blank mapping strings using NULLIF.
      - Uses safe LIKE matching instead of RLIKE to prevent regex failures.
   6. NAME BRIDGE: Bridges the Mapping Table's String Name to the Master List's 
      Numeric BusinessID.
   7. FINAL OUTPUT: If a mapped AU has 1 or more flagged engagements, outputs 'Yes', 
      otherwise safely defaults to 'No'. Maintains strict 6-column structure.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 5 & 6. Grab valid TPRM Mapping Strings
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3. Apply Comma Exceptions
    SELECT 
        EngagementNumber,
        
        -- EXCEPTION DICTIONARY: Protect known LOBs that have internal commas
        REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience') AS safe_lob_using,
            
        KPI_Number,
        KPI_Value,
        primary_product_serv
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flagged_Engagements AS (
    -- 2. Filter the TP data based on the KPI rules OR the specific service categories
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using
    FROM Third_Party_Risk_Data
    WHERE 
        -- Condition 1: KPI 8.5 equals 100
        (TRIM(KPI_Number) = '8.5' AND TRIM(KPI_Value) = '100')
        
        OR 
        
        -- Condition 2: Case-Insensitive match to Target Services
        UPPER(TRIM(primary_product_serv)) IN (
            UPPER('Marketing media services'),
            UPPER('Marketing and distribution'),
            UPPER('Direct marketing print service'),
            UPPER('Print advertising'),
            UPPER('Printing'),
            UPPER('Management and Business Professionals and Administrative Services'),
            UPPER('Marketing analysis'),
            UPPER('Publicity and marketing support services'),
            UPPER('Marketing communications agencies'),
            UPPER('Advertising agency services'),
            UPPER('Business forms or questionnaires'),
            UPPER('Specialized warehousing and storage'),
            UPPER('Stationery or business form printing'),
            UPPER('Published Products')
        )
),

Flattened_LOBs AS (
    -- 4. FLATTEN RULE: Split by commas, expand into individual rows, and restore the protected commas
    SELECT 
        EngagementNumber, 
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 5 & 6. Bridge mappings to Master List and count the matches
    SELECT 
        mast.BusinessID,
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_AU_Name)) = UPPER(TRIM(mast.AU_Name))
    GROUP BY mast.BusinessID
)

-- 7. Final Output: Strict 6-column structure anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP03' AS QuestionID,               
    
    -- Yes/No Logic based on whether they had any matching engagements
    CASE 
        WHEN COALESCE(e.Match_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP03 - Service Category Traceability
   
   PURPOSE: Verifies the KPI/Service flag logic, prints the full comma-separated LOB 
   strings vs the exploded chunks (including comma protection), and displays the 
   Master AU lookup status to troubleshoot mapping drops.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Filtered_TP_Data AS (
    SELECT 
        p.EngagementNumber,
        p.KPI_Number,
        p.Value,
        p.primary_product_serv,
        p.owning_lob AS Raw_Col_K,
        p.lob_using AS Raw_Col_L,
        
        -- Apply the protection layer for the debug view
        REPLACE(p.owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_owning_lob,
        REPLACE(p.lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_lob_using
        
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    WHERE p.EngagementNumber IS NOT NULL
      AND (
          (TRIM(p.KPI_Number) = '8.5' AND TRIM(p.Value) = '100') OR 
          TRIM(p.primary_product_serv) IN (
              'Marketing media services', 'Marketing and distribution', 'Direct marketing print service',
              'Print advertising', 'Printing', 'Management and Business Professionals and Administrative Services',
              'Marketing analysis', 'Publicity and marketing support services', 'Marketing communications agencies',
              'Advertising agency services', 'Business forms or questionnaires', 'Specialized warehousing and storage',
              'Stationery or business form printing', 'Published Products'
          )
      )
),

Flattened AS (
    SELECT 
        f.EngagementNumber,
        f.KPI_Number,
        f.Value,
        f.primary_product_serv,
        f.Raw_Col_K,
        f.Raw_Col_L,
        f.safe_owning_lob,
        
        -- Restore the string for the final exploded chunk
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Exploded_Chunk
    FROM Filtered_TP_Data f
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

SELECT DISTINCT
    f.EngagementNumber,
    
    -- Service Flag Traceability
    f.KPI_Number,
    f.Value,
    f.primary_product_serv AS Col_F_Service,
    
    -- LOB Flattening Traceability
    f.Raw_Col_K AS Original_String_From_DB,
    f.safe_owning_lob AS String_With_Comma_Protection,
    f.Exploded_Chunk AS Final_Restored_Chunk,
    
    -- Mapping & Bridging Traceability
    map.TPRM_Units AS Matched_Mapping_String,
    map.Assessable_Unit_ID AS Mapping_Table_AU_Name,
    mast.Master_Numeric_ID AS Bridged_Master_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'MISSING FROM MASTER LIST' ELSE 'SUCCESSFULLY BRIDGED' END AS Master_List_Status
    
FROM Flattened f
-- Safe Mapping Join
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(f.Exploded_Chunk) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
-- Name Bridge Join
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
ORDER BY f.EngagementNumber;